# Selekcja datasetu — wydzielenie podzbioru treningowego

Notebook implementuje sekwencyjną filtrację i eksport tabeli decyzyjnej.

**Wejście:** `processed_texts.csv` (z `analysis.ipynb`), `pjm_data.csv`, `texts_eng.h5`  
**Wyjście:** `train_selection.csv`, `selection_analysis.csv`

## Uruchomienie w Google Colab

Wgraj `processed_texts.csv`, `pjm_data.csv`, `texts_eng.h5` i `hdf5_manager.py` do sesji, a potem uruchom poniższą komórkę.

In [1]:
import subprocess
import sys


def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


if _in_colab():
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "pandas",
            "plotly",
            "h5py",
            "numpy",
            "transformers",
            "sentencepiece",
            "huggingface_hub",
        ]
    )
    print("Colab: zależności gotowe.")


In [2]:
import ast
import json as _json
import numpy as np
import pandas as pd
import plotly.express as px
from collections import Counter

df = pd.read_csv("./data/processed_texts.csv")

if isinstance(df["nlp"].iloc[0], str):
    # JSON is ~10x faster than ast.literal_eval for nested dicts
    try:
        df["nlp"] = df["nlp"].apply(_json.loads)
    except Exception:
        df["nlp"] = df["nlp"].apply(ast.literal_eval)

if "source_flag" not in df.columns:
    from hdf5_manager import HDF5Loader

    hdf5 = HDF5Loader("./data/texts_eng.h5")
    hdf5_flags = hdf5.load_flags()
    hdf5.close()
    df["source_flag"] = df["key"].map(hdf5_flags)
    print(f"Flagi źródła: {df['source_flag'].value_counts().to_string()}")

if "speaker_id" not in df.columns:
    df_meta = pd.read_csv("./data/pjm_data.csv")
    df["video_id"] = df["key"].str.split("_").str[0].astype(int)
    _side_map = {"L": "left", "R": "right", "M": "-"}
    df["speaker_side"] = df["key"].str.split("_").str[2].map(_side_map)
    n_before = len(df)
    df = df.merge(
        df_meta[["video_id", "speaker_side", "speaker_id", "sex", "age", "task_id", "video_type", "special_case"]],
        on=["video_id", "speaker_side"],
        how="left",
    )
    assert len(df) == n_before, f"Merge zmienił liczbę wierszy: {n_before} -> {len(df)}"

if "interaction_type" not in df.columns:
    df["interaction_type"] = df["video_type"].str.split(";").str[0]

if "mask_ratio" not in df.columns:
    df["mask_ratio"] = df["masked"].apply(
        lambda x: str(x).count("<") / max(len(str(x).split()), 1)
    )
if "final_wc" not in df.columns:
    df["final_wc"] = df["masked"].str.split().str.len()

word_freq: Counter[str] = Counter()
for text in df["lemmatized"]:
    word_freq.update(str(text).split())

total_occurrences = sum(word_freq.values())
cumsum = 0.0
word_vocab_pct: dict[str, float] = {}
for word, count in word_freq.most_common():
    cumsum += count
    word_vocab_pct[word] = round(cumsum / total_occurrences * 100, 2)

df["min_vocab_pct"] = df["lemmatized"].apply(
    lambda t: max((word_vocab_pct.get(w, 100.0) for w in str(t).split()), default=0.0)
)

print(f"Załadowano {len(df)} Nagrań, {df['speaker_id'].nunique()} migających")
print(f"min_vocab_pct percentyle: {df['min_vocab_pct'].describe()[['25%','50%','75%','max']].to_string()}")

Załadowano 39817 Nagrań, 151 migających
min_vocab_pct percentyle: 25%     76.91
50%     88.83
75%     95.58
max    100.00


### Faza 1 — Diagnoza stanu surowego

In [3]:
total = len(df)
empty = (df["final_wc"] == 0).sum()
single = (df["word_count"] == 1).sum()

key_dupes = df["key"].duplicated().sum()
global_lemma_dupes = df["lemmatized"].duplicated().sum()

print("=" * 60)
print("DIAGNOZA STANU SUROWEGO")
print("=" * 60)
print(f"Wpisy ogółem:                          {total}")
print(f"Puste (final_wc == 0):                 {empty} ({empty/total*100:.1f}%)")
print(f"Jednowyrazowe (word_count == 1):       {single} ({single/total*100:.1f}%)")
print()
print(f"Duplikaty po key (dosłowne kopie):     {key_dupes}")
print(f"Wspólne lematy (różne segmenty wideo): {global_lemma_dupes} ({global_lemma_dupes/total*100:.1f}%)")
print("  → Różne wejście wizualne, ten sam tekst — wartościowe dla treningu")
print()
print("Źródło tłumaczenia (source_flag z HDF5):")
print(df["source_flag"].value_counts().to_string())
print()
print("Typ interakcji:")
print(df["interaction_type"].value_counts().to_string())
print()
special = df["special_case"].value_counts()
print(f"Special case: {special.to_string()}")
one_hand_n = (df["special_case"] == "one_hand").sum()
if one_hand_n > 0:
    print(f"  → {one_hand_n} segmentów od jednoręcznego migacza — flaga do analizy wideo")
print()
print(f"Unikalni migający (speaker_id):       {df['speaker_id'].nunique()}")
print(f"Brak speaker_id:                     {df['speaker_id'].isna().sum()}")
print()
print("mask_ratio — percentyle:")
for p in [25, 50, 75, 90, 95, 99]:
    print(f"  P{p:2d}: {df['mask_ratio'].quantile(p / 100):.3f}")
print()
print("word_count — statystyki:")
print(df["word_count"].describe().to_string())
print(f"\n>32 tokenów: {(df['word_count'] > 32).sum()}")

DIAGNOZA STANU SUROWEGO
Wpisy ogółem:                          39817
Puste (final_wc == 0):                 1 (0.0%)
Jednowyrazowe (word_count == 1):       5530 (13.9%)

Duplikaty po key (dosłowne kopie):     0
Wspólne lematy (różne segmenty wideo): 9366 (23.5%)
  → Różne wejście wizualne, ten sam tekst — wartościowe dla treningu

Źródło tłumaczenia (source_flag z HDF5):
source_flag
deepl      39749
manual        58
invalid       10

Typ interakcji:
interaction_type
dialog     25602
monolog    14215

Special case: special_case
-           39539
one_hand      278
  → 278 segmentów od jednoręcznego migacza — flaga do analizy wideo

Unikalni migający (speaker_id):       151
Brak speaker_id:                     0

mask_ratio — percentyle:
  P25: 0.000
  P50: 0.000
  P75: 0.000
  P90: 0.000
  P95: 0.100
  P99: 0.333

word_count — statystyki:
count    39817.000000
mean         7.149434
std          6.039138
min          1.000000
25%          3.000000
50%          6.000000
75%         10.0000

In [4]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

MASK_RATIO_THRESHOLD = 0.3

zero_pct = (df["mask_ratio"] == 0).mean() * 100
nonzero = df[df["mask_ratio"] > 0]["mask_ratio"]

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        f"mask_ratio > 0 ({len(nonzero)} zdań, {100 - zero_pct:.1f}% korpusu)",
        "Skumulowany rozkład (CDF)",
    ],
    horizontal_spacing=0.12,
)

fig.add_trace(
    go.Histogram(x=nonzero, nbinsx=40, name="mask_ratio > 0"),
    row=1, col=1,
)
fig.update_xaxes(title_text="mask_ratio", row=1, col=1)
fig.update_yaxes(title_text="Liczba zdań", row=1, col=1)

sorted_mr = df["mask_ratio"].sort_values()
fig.add_trace(
    go.Scatter(
        x=sorted_mr.values,
        y=np.arange(1, len(sorted_mr) + 1) / len(sorted_mr) * 100,
        mode="lines", name="CDF",
    ),
    row=1, col=2,
)
fig.add_hline(y=100 - zero_pct, line_dash="dot", row=1, col=2,
              annotation_text=f"{zero_pct:.0f}% ma mask_ratio=0")
fig.add_vline(x=MASK_RATIO_THRESHOLD, line_dash="dash", row=1, col=2,
              annotation_text=f"próg={MASK_RATIO_THRESHOLD}")
fig.update_xaxes(title_text="mask_ratio", row=1, col=2)
fig.update_yaxes(title_text="% zdań ≤ próg", row=1, col=2)

fig.update_layout(
    title_text=f"Rozkład mask_ratio — {zero_pct:.1f}% zdań nie ma żadnych masek",
    showlegend=False, height=400,
)
fig.show()

entries_per_speaker = df["speaker_id"].value_counts()
fig2 = px.histogram(
    x=entries_per_speaker.values, nbins=30,
    title="Rozkład liczby wpisów per migacz",
    labels={"x": "Nagrań", "y": "Migaczy"},
)
fig2.update_xaxes(title_text="Nagrań per migacz")
fig2.update_yaxes(title_text="Liczba migaczy")
fig2.show()

### Faza 2 — Reguły filtracji (keep / drop / review)

Sekwencyjne filtry z licznikiem po każdym kroku.  
Kolejność ma znaczenie — każdy filtr działa tylko na wierszach, które przeżyły poprzednie.

In [5]:
from transformers import AutoTokenizer

MASK_RATIO_THRESHOLD = 0.3

df["decision"] = "keep"
df["reason"] = "keep"
df["flag"] = ""

filter_log: list[tuple[str, int, int]] = []


def apply_filter(name: str, mask: "pd.Series[bool]") -> None:
    active = (df["decision"] == "keep") & mask
    df.loc[active, "decision"] = "drop"
    df.loc[active, "reason"] = name
    filter_log.append((name, int(active.sum()), int((df["decision"] == "keep").sum())))


# 1. Oznaczone jako invalid w HDF5
apply_filter("invalid_source", df["source_flag"] == "invalid")

# 2. Puste / brak treści
apply_filter("empty", (df["final_wc"] == 0) | df["clean"].str.fullmatch(r"[\d\s]*", na=True))

# 3. Jednowyrazowe
apply_filter("single_word", df["word_count"] == 1)

# 4. Wysoki mask_ratio
apply_filter("high_mask_ratio", df["mask_ratio"] > MASK_RATIO_THRESHOLD)

# 5. Przekraczające okno dekodera T5 (max_txt_len=64 subtokenów)

t5_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
assert t5_tokenizer is not None
df["subtoken_len"] = df["text"].apply(lambda x: len(t5_tokenizer.encode(str(x))))  # type: ignore[union-attr]
apply_filter("exceeds_decoder_window", df["subtoken_len"] > 60)

# --- Podsumowanie ---
print(f"{'Filtr':<32} {'Dropped':>8} {'Remaining':>10}")
print("-" * 52)
for name, dropped, remaining in filter_log:
    print(f"{name:<32} {dropped:>8} {remaining:>10}")
print("=" * 52)
kept_total = int((df["decision"] == "keep").sum())
drop_total = int((df["decision"] == "drop").sum())
print(f"{'TOTAL':<32} {drop_total:>8} {kept_total:>10}")
print(f"\nKeep rate: {kept_total / len(df) * 100:.1f}%")

Filtr                             Dropped  Remaining
----------------------------------------------------
invalid_source                         10      39807
empty                                   2      39805
single_word                          5527      34278
high_mask_ratio                       362      33916
exceeds_decoder_window                 19      33897
TOTAL                                5920      33897

Keep rate: 85.1%


In [7]:
# Flagowanie (wektoryzowane — bez apply axis=1)

# Wyciągamy sety encji raz, vectorially
ent_sets = df["nlp"].apply(lambda x: {t.lower() for t, _ in x.get("ents", [])} if isinstance(x, dict) else set())

# NER-flagging: maska kandydatów + sprawdzenie per-wiersz tylko dla nich (mały subset)
ner_candidates = (
    (df["decision"] == "keep")
    & (df["mask_ratio"] > 0)
    & (df["mask_ratio"] <= 0.15)
    & (ent_sets.str.len() > 0)  # tylko wiersze z jakimiś encjami
)

def _has_only_ner_masks(lemma_str, masked_str, ents):
    for lw, mw in zip(str(lemma_str).split(), str(masked_str).split()):
        if mw.startswith("<") and mw.endswith(">"):
            if lw not in ents:
                return False
    return True

# apply tylko na kandydatach (~ułamek datasetu, nie 33k wierszy)
cand_idx = df.index[ner_candidates]
ner_result = pd.Series(False, index=df.index)
ner_result.loc[cand_idx] = [
    _has_only_ner_masks(df.at[i, "lemmatized"], df.at[i, "masked"], ent_sets.at[i])
    for i in cand_idx
]

if "flag" not in df.columns:
    df["flag"] = ""

df.loc[ner_result, "flag"] = "has_rare_ner"

short_mask = (df["decision"] == "keep") & df["word_count"].between(2, 3)
df.loc[short_mask & (df["flag"] == ""), "flag"] = "short"
df.loc[short_mask & (df["flag"] != "") & (~df["flag"].str.contains("short")), "flag"] += ";short"

one_hand_mask = (df["decision"] == "keep") & (df["special_case"] == "one_hand")
df.loc[one_hand_mask & (df["flag"] == ""), "flag"] = "one_hand"
df.loc[one_hand_mask & (df["flag"] != "") & (~df["flag"].str.contains("one_hand")), "flag"] += ";one_hand"

print(f"NER-flagged (kept, mask_ratio<=0.15, only NER masks): {ner_result.sum()}")
print(f"Short-flagged (kept, 2-3 words):                      {short_mask.sum()}")
print(f"One-hand-flagged (kept, jednoręczny migający):         {one_hand_mask.sum()}")
print("Rozkład decyzji:")
print(df["decision"].value_counts().to_string())
print("Rozkład powodów drop:")
print(df.loc[df["decision"] == "drop", "reason"].value_counts().to_string())


NER-flagged (kept, mask_ratio<=0.15, only NER masks): 0
Short-flagged (kept, 2-3 words):                      6688
One-hand-flagged (kept, jednoręczny migający):         230
Rozkład decyzji:
decision
keep    33897
drop     5920
Rozkład powodów drop:
reason
single_word               5527
high_mask_ratio            362
exceeds_decoder_window      19
invalid_source              10
empty                        2


/tmp/ipykernel_766501/2085922269.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[]' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  ner_result.loc[cand_idx] = [


### Faza 3 — Czułość progów (argumentacja do pracy)

Jak zmieniają się wyniki filtracji przy różnych wartościach `mask_ratio` i `THRESHOLD`?

In [8]:
# Czułość na mask_ratio
ratios = np.arange(0.0, 1.01, 0.05)
survive_pct = []
for r in ratios:
    base_drop = (
        (df["final_wc"] == 0)
        | (df["word_count"] == 1)
        | (df["mask_ratio"] > r)
    )
    survive_pct.append((1 - base_drop.mean()) * 100)

fig = px.line(
    x=ratios, y=survive_pct,
    title="Czułość: próg mask_ratio vs % zdań przeżywających",
    labels={"x": "Próg mask_ratio", "y": "% zdań (keep)"},
)
fig.add_vline(x=MASK_RATIO_THRESHOLD, line_dash="dash", annotation_text=f"wybrany: {MASK_RATIO_THRESHOLD}")
fig.show()

# Czułość na THRESHOLD (min. częstość słowa do zbudowania rare_words)
thresholds = [1, 2, 3, 5, 10, 20]
rows = []
for thr in thresholds:
    rare = {w for w, c in word_freq.items() if c <= thr}
    n_rare = len(rare)
    mr = df["lemmatized"].apply(
        lambda t: sum(1 for w in str(t).split() if w in rare) / max(len(str(t).split()), 1)
    )
    survive = ((mr <= 0.3) & (df["word_count"] > 1) & (df["final_wc"] > 0)).mean() * 100
    rows.append({"THRESHOLD": thr, "rare_words": n_rare, "survive_%": round(survive, 1)})

sens_df = pd.DataFrame(rows)
print(sens_df.to_string(index=False))

fig2 = px.bar(
    sens_df, x="THRESHOLD", y="survive_%",
    title="Czułość: THRESHOLD (min. częstość) vs % zdań przeżywających",
    text="survive_%",
)
fig2.update_yaxes(title_text="% zdań (keep)")
fig2.show()

 THRESHOLD  rare_words  survive_%
         1        1622       85.8
         2        2184       85.5
         3        2541       85.2
         5        2933       84.7
        10        3375       83.8
        20        3699       82.5


### Faza 4 — Profil przefiltrowanego zbioru

Porównanie rozkładów przed i po filtracji — czy filtr nie przycina nierównomiernie.

In [9]:
kept = df[df["decision"] == "keep"].copy()
dropped = df[df["decision"] == "drop"].copy()

# 1. Rozkład word_count: oryginał vs przefiltrowany
fig = px.histogram(
    pd.DataFrame({
        "Oryginał": df["word_count"],
        "Po filtracji": kept["word_count"].reindex(df.index),
    }),
    barmode="overlay", nbins=40, opacity=0.6,
    title="Rozkład word_count: oryginał vs po filtracji",
)
fig.update_xaxes(title_text="word_count")
fig.update_yaxes(title_text="Liczba")
fig.show()

# 2. Pokrycie słownictwa: kumulacja lematów
freq_all = Counter()
for t in df["lemmatized"]:
    freq_all.update(str(t).split())

freq_kept = Counter()
for t in kept["lemmatized"]:
    freq_kept.update(str(t).split())

def cumulative_coverage(freq: Counter) -> list[float]:
    total = sum(freq.values())
    cumsum = 0.0
    coverage = []
    for _, count in freq.most_common():
        cumsum += count
        coverage.append(cumsum / total * 100)
    return coverage

cov_all = cumulative_coverage(freq_all)
cov_kept = cumulative_coverage(freq_kept)

fig2 = px.line(
    title="Kumulacyjne pokrycie słownictwa (lematy)",
    labels={"x": "Liczba unikalnych lematów", "y": "% pokrycia korpusu"},
)
fig2.add_scatter(x=list(range(len(cov_all))), y=cov_all, name="Oryginał")
fig2.add_scatter(x=list(range(len(cov_kept))), y=cov_kept, name="Po filtracji")
fig2.show()

print(f"Unikalne lematy — oryginał: {len(freq_all)}, po filtracji: {len(freq_kept)}")
print(f"Utracone lematy: {len(freq_all) - len(freq_kept)}")

Unikalne lematy — oryginał: 4661, po filtracji: 4327
Utracone lematy: 334


### Faza 5 — Eksport

Dwa pliki CSV:
- **`train_selection.csv`** — tylko `keep`, minimalne kolumny do stratyfikacji (speaker_id, interaction_type, sex, age, flag)
- **`selection_analysis.csv`** — pełny zbiór (keep + drop) ze wszystkimi metadanymi, do dalszej analizy

In [10]:
from IPython.display import display as ipy_display

train_cols = ["key", "speaker_id", "interaction_type", "sex", "age", "min_vocab_pct", "decision", "reason", "flag"]
train_df = df.loc[df["decision"] == "keep", train_cols].copy()
train_path = "./data/train_selection.csv"
train_df.to_csv(train_path, index=False)

analysis_cols = [
    "key", "speaker_id", "source_flag", "interaction_type", "video_type",
    "sex", "age", "special_case", "decision", "reason", "text",
    "word_count", "mask_ratio", "subtoken_len", "min_vocab_pct", "flag",
]
analysis_df = df[[c for c in analysis_cols if c in df.columns]].copy()
analysis_path = "./data/selection_analysis.csv"
analysis_df.to_csv(analysis_path, index=False)

print(f"Plik treningowy:  {train_path}  ({len(train_df)} wierszy, {len(train_cols)} kolumn)")
print(f"Plik analityczny: {analysis_path}  ({len(analysis_df)} wierszy, {len(analysis_df.columns)} kolumn)")
print()
print("Podsumowanie decyzji:")
print(f"  keep:  {(df['decision'] == 'keep').sum()}")
print(f"  drop:  {(df['decision'] == 'drop').sum()}")
print()
print("Powody drop:")
print(df.loc[df["decision"] == "drop", "reason"].value_counts().to_string())
print()
flags = train_df["flag"]
print("Flagi (kept):")
print(flags[flags != ""].value_counts().to_string() if (flags != "").any() else "  (brak flag)")
print()
print("--- train_selection.csv ---")
ipy_display(train_df.head(10))
print("\n--- selection_analysis.csv ---")
ipy_display(analysis_df.head(10))

Plik treningowy:  ./data/train_selection.csv  (33897 wierszy, 9 kolumn)
Plik analityczny: ./data/selection_analysis.csv  (39817 wierszy, 16 kolumn)

Podsumowanie decyzji:
  keep:  33897
  drop:  5920

Powody drop:
reason
single_word               5527
high_mask_ratio            362
exceeds_decoder_window      19
invalid_source              10
empty                        2

Flagi (kept):
flag
short             6631
one_hand           173
short;one_hand      57

--- train_selection.csv ---


,key,speaker_id,interaction_type,sex,age,min_vocab_pct,decision,reason,flag
0,001_000_M,F_46-60_001,monolog,F,46-60,73.47,keep,keep,
1,001_001_M,F_46-60_001,monolog,F,46-60,76.99,keep,keep,
2,001_002_M,F_46-60_001,monolog,F,46-60,96.95,keep,keep,
3,001_003_M,F_46-60_001,monolog,F,46-60,73.47,keep,keep,
4,001_004_M,F_46-60_001,monolog,F,46-60,91.48,keep,keep,
5,001_005_M,F_46-60_001,monolog,F,46-60,58.04,keep,keep,
6,001_006_M,F_46-60_001,monolog,F,46-60,83.95,keep,keep,short
7,001_007_M,F_46-60_001,monolog,F,46-60,83.31,keep,keep,short
8,001_008_M,F_46-60_001,monolog,F,46-60,79.21,keep,keep,
9,001_009_M,F_46-60_001,monolog,F,46-60,57.55,keep,keep,short



--- selection_analysis.csv ---


,key,speaker_id,source_flag,interaction_type,video_type,sex,age,special_case,decision,reason,text,word_count,mask_ratio,subtoken_len,min_vocab_pct,flag
0,001_000_M,F_46-60_001,deepl,monolog,monolog;obraz;komiks;Frog Story,F,46-60,-,keep,keep,There is a frog in a glass jar.,8,0.0,15,73.47,
1,001_001_M,F_46-60_001,deepl,monolog,monolog;obraz;komiks;Frog Story,F,46-60,-,keep,keep,The dog looks inside.,4,0.0,6,76.99,
2,001_002_M,F_46-60_001,deepl,monolog,monolog;obraz;komiks;Frog Story,F,46-60,-,keep,keep,The child lies on the bed and supports his hea...,13,0.0,15,96.95,
3,001_003_M,F_46-60_001,deepl,monolog,monolog;obraz;komiks;Frog Story,F,46-60,-,keep,keep,"I said, the dog looks down at the frog sitting...",13,0.0,19,73.47,
4,001_004_M,F_46-60_001,deepl,monolog,monolog;obraz;komiks;Frog Story,F,46-60,-,keep,keep,"In the second picture, the girl is sleeping un...",11,0.0,14,91.48,
5,001_005_M,F_46-60_001,deepl,monolog,monolog;obraz;komiks;Frog Story,F,46-60,-,keep,keep,The frog is gone.,4,0.0,8,58.04,
6,001_006_M,F_46-60_001,deepl,monolog,monolog;obraz;komiks;Frog Story,F,46-60,-,keep,keep,She escaped.,2,0.0,5,83.95,short
7,001_007_M,F_46-60_001,deepl,monolog,monolog;obraz;komiks;Frog Story,F,46-60,-,keep,keep,"She disappeared,",2,0.0,4,83.31,short
8,001_008_M,F_46-60_001,deepl,monolog,monolog;obraz;komiks;Frog Story,F,46-60,-,keep,keep,The dog calls the frog out the window.,8,0.0,12,79.21,
9,001_009_M,F_46-60_001,deepl,monolog,monolog;obraz;komiks;Frog Story,F,46-60,-,keep,keep,She doesn't.,2,0.0,6,57.55,short


### Faza 6 (plan na później) — Co zrobić z wideo

Poniższe kroki **nie są implementowane** w tym notebooku — to plan do kolejnego etapu.

1. **Korelacja tekst ↔ wideo**  
   Załadować metadane PJM (czas trwania segmentów), dołączyć do tabeli decyzyjnej.  
   Scatter: `word_count` vs `duration_sec`. Outliery przez odchylenie od regresji (nie globalny próg).

2. **Sklejanie krótkich segmentów**  
   Zdefiniować sąsiedztwo (ten sam nagranie-źródło + następny segment czasowy).  
   Scalić wideo + tekst atomowo. Przeliczyć filtry na nowej jednostce.  
   Po sklejeniu: decyzja keep/drop na poziomie **całego** złożonego segmentu.

3. **Grupowy split train/val/test**  
   Podział po `speaker_id` — żadna osoba migająca nie trafia jednocześnie do train i test.  
   Zapobiega przeciekowi stylu migania między zbiorami.


### Faza 6a — Audit nierównomierności migających

Ile segmentów pochodzi od każdej osoby migającej? Czy kilka dominujących osób zdominuje trening?


In [11]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

train_df = pd.read_csv("./data/train_selection.csv")
speaker_counts = train_df["speaker_id"].value_counts().sort_values(ascending=False)

total_segs = len(train_df)
total_speakers = len(speaker_counts)

print("=" * 60)
print("AUDIT NIERÓWNOMIERNOŚCI MIGAJĄCYCH")
print("=" * 60)
print(f"Łączna liczba segmentów (keep): {total_segs}")
print(f"Unikalnych migających:             {total_speakers}")
print(f"Średnio segmentów/os. migającej:      {speaker_counts.mean():.1f}")
print(f"Mediana:                        {speaker_counts.median():.0f}")
print(f"Min / Max:                      {speaker_counts.min()} / {speaker_counts.max()}")
print()

# Ilu migaczy pokrywa X% datasetu
cumulative_pct = speaker_counts.cumsum() / total_segs * 100
for threshold in [50, 70, 80, 90]:
    n = int((cumulative_pct <= threshold).sum()) + 1
    print(f"Os. migających pokrywających {threshold}% datasetu: {n} z {total_speakers} ({n/total_speakers*100:.1f}%)")

print()

# Top 10 / Bottom 10
print("Top 10 osób migających (najwięcej segmentów):")
print(speaker_counts.head(10).to_string())
print()
print("Bottom 10 osób migających (najmniej segmentów):")
print(speaker_counts.tail(10).to_string())

# Wizualizacja
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Rozkład segmentów per osoba migająca", "Krzywa koncentracji (Lorenza)"],
    horizontal_spacing=0.12,
)

fig.add_trace(
    go.Histogram(x=speaker_counts.values, nbinsx=30, name="Migający"),
    row=1, col=1,
)
fig.update_xaxes(title_text="Segmentów", row=1, col=1)
fig.update_yaxes(title_text="Liczba migających", row=1, col=1)

# Krzywa Lorenza
sorted_counts = np.sort(speaker_counts.values)
cum_share_speakers = np.arange(1, len(sorted_counts) + 1) / len(sorted_counts) * 100
cum_share_segs = np.cumsum(sorted_counts) / sorted_counts.sum() * 100

fig.add_trace(
    go.Scatter(x=cum_share_speakers, y=cum_share_segs, mode="lines", name="Lorenz"),
    row=1, col=2,
)
fig.add_trace(
    go.Scatter(x=[0, 100], y=[0, 100], mode="lines", line=dict(dash="dash", color="gray"),
               name="Równość idealna"),
    row=1, col=2,
)
fig.update_xaxes(title_text="% migających (skumulowany)", row=1, col=2)
fig.update_yaxes(title_text="% segmentów (skumulowany)", row=1, col=2)

fig.update_layout(title_text="Nierównomierność datasetu per osoba migająca", height=420, showlegend=False)
fig.show()

# Analiza kapowania (proportion cap)
print("\nWpływ kapowania proporcji migających na rozmiar datasetu:")
print(f"{'Cap (% datasetu)':>18} {'Cap (seg/mig)':>14} {'Segm. po kap.':>14} {'Retencja':>10}")
print("-" * 60)
for cap_pct in [0.5, 1.0, 1.5, 2.0, 3.0, 5.0, 100.0]:
    cap = int(total_segs * cap_pct / 100)
    retained = int(speaker_counts.clip(upper=cap).sum())
    print(f"{cap_pct:>17.1f}% {cap:>14} {retained:>14} {retained/total_segs*100:>9.1f}%")


AUDIT NIERÓWNOMIERNOŚCI MIGAJĄCYCH
Łączna liczba segmentów (keep): 33897
Unikalnych migających:             151
Średnio segmentów/os. migającej:      224.5
Mediana:                        211
Min / Max:                      29 / 532

Os. migających pokrywających 50% datasetu: 51 z 151 (33.8%)
Os. migających pokrywających 70% datasetu: 81 z 151 (53.6%)
Os. migających pokrywających 80% datasetu: 99 z 151 (65.6%)
Os. migających pokrywających 90% datasetu: 121 z 151 (80.1%)

Top 10 osób migających (najwięcej segmentów):
speaker_id
F_32-45_001    532
F_18-30_007    530
M_60+_013      527
F_46-60_001    461
M_18-30_002    451
F_60+_011      450
F_46-60_014    435
F_32-45_024    406
M_32-45_023    404
M_46-60_014    401

Bottom 10 osób migających (najmniej segmentów):
speaker_id
M_32-45_019    104
M_32-45_010    102
F_46-60_023     95
F_60+_006       92
F_32-45_019     88
M_46-60_010     85
M_32-45_015     49
F_60+_002       43
M_60+_012       29
M_60+_018       29



Wpływ kapowania proporcji migających na rozmiar datasetu:
  Cap (% datasetu)  Cap (seg/mig)  Segm. po kap.   Retencja
------------------------------------------------------------
              0.5%            169          23414      69.1%
              1.0%            338          32461      95.8%
              1.5%            508          33832      99.8%
              2.0%            677          33897     100.0%
              3.0%           1016          33897     100.0%
              5.0%           1694          33897     100.0%
            100.0%          33897          33897     100.0%


### Faza 6b — Jakość kadrowania (crop_params.lmdb)

Analiza geometryczna ramek z YOLO: pole powierzchni, proporcje, wartości paddingu.
Nie ma explicit 'confidence' — jakość wyprowadzamy z geometrii boxu.


In [12]:
import lmdb
import msgpack

LMDB_PATH = "./data/crop_params.lmdb"  # output of step 05

env = lmdb.open(LMDB_PATH, readonly=True, lock=False)
crop_data = []
with env.begin() as txn:
    for key_bytes, val_bytes in txn.cursor():
        key = key_bytes.decode()
        val = msgpack.unpackb(val_bytes)
        w = val["x_end"] - val["x_start"]
        h = val["y_end"] - val["y_start"]
        pad = val["padding"]
        total_pad = sum(pad.values())
        crop_data.append({
            "key": key,
            "box_width": w,
            "box_height": h,
            "box_area": w * h,
            "aspect_ratio": w / max(h, 1),  # width/height; typowy migający: ~0.5-0.7
            "total_padding": total_pad,
            "has_padding": total_pad > 0,
        })
env.close()

crop_df = pd.DataFrame(crop_data)
print(f"Kluczy w LMDB: {len(crop_df):,}")
print()

# Statystyki ogólne
print("Statystyki geometrii boxów:")
print(crop_df[["box_width", "box_height", "box_area", "aspect_ratio", "total_padding"]].describe().round(1).to_string())
print()

# Progi jakości — heurystyczne
AREA_LOW = 10_000   # ~100x100 px — zbyt małe
AREA_HIGH = 400_000 # ~600x600+ px — nieprawdopodobnie duże
AR_LOW, AR_HIGH = 0.25, 2.0  # poza typowym zakresem sylwetki

bad_area = ((crop_df["box_area"] < AREA_LOW) | (crop_df["box_area"] > AREA_HIGH)).sum()
bad_ar   = ((crop_df["aspect_ratio"] < AR_LOW) | (crop_df["aspect_ratio"] > AR_HIGH)).sum()
with_pad = crop_df["has_padding"].sum()

print(f"Podejrzane boksy (pole < {AREA_LOW:,} lub > {AREA_HIGH:,} px²): {bad_area} ({bad_area/len(crop_df)*100:.2f}%)")
print(f"Podejrzane proporcje (AR < {AR_LOW} lub > {AR_HIGH}):            {bad_ar}  ({bad_ar/len(crop_df)*100:.2f}%)")
print(f"Segmenty z paddingiem (≥1 bok):                                  {with_pad} ({with_pad/len(crop_df)*100:.1f}%)")
print()

# Jak wiele kluczy z train_selection pokrywa LMDB?
lmdb_keys = set(crop_df["key"])
train_keys = set(train_df["key"])
overlap = train_keys & lmdb_keys
missing = train_keys - lmdb_keys
print(f"Klucze w train_selection:        {len(train_keys):,}")
print(f"Klucze w LMDB:                   {len(lmdb_keys):,}")
print(f"Pokrycie (LMDB ∩ train):         {len(overlap):,} ({len(overlap)/len(train_keys)*100:.2f}%)")
print(f"Brakujące w LMDB (PRD: ~4 seg.): {len(missing)}")
if missing:
    print("  Przykłady:", list(missing)[:8])
print()

# Wizualizacja rozkładów
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Rozkład pola boxu (px²)", "Rozkład proporcji (width/height)"],
)
fig.add_trace(go.Histogram(x=crop_df["box_area"], nbinsx=50, name="Pole"), row=1, col=1)
fig.add_trace(go.Histogram(x=crop_df["aspect_ratio"], nbinsx=50, name="AR"), row=1, col=2)
fig.update_layout(title_text="Geometria boxów crop_params.lmdb", height=380, showlegend=False)
fig.show()

# Per-speaker crop stats (merge by key prefix video_id_segidx_side)
crop_df["speaker_key"] = crop_df["key"].apply(lambda k: "_".join(k.split("_")[:1]))
# Join with train_df on key to get speaker_id
crop_train = crop_df[crop_df["key"].isin(train_keys)].merge(
    train_df[["key", "speaker_id"]], on="key", how="left"
)
per_speaker_quality = crop_train.groupby("speaker_id").agg(
    n_segs=("key", "count"),
    mean_area=("box_area", "mean"),
    mean_ar=("aspect_ratio", "mean"),
    pct_padded=("has_padding", "mean"),
).round(1)
print("Statystyki jakości kadrowania per osoba migająca (top 15 wg udziału z paddingiem):")
print(per_speaker_quality.sort_values("pct_padded", ascending=False).head(15).to_string())


Kluczy w LMDB: 39,813

Statystyki geometrii boxów:
       box_width  box_height  box_area  aspect_ratio  total_padding
count    39813.0     39813.0   39813.0       39813.0        39813.0
mean       396.1       387.4  158261.4           1.0            9.9
std         72.9        71.8   65976.9           0.1           29.0
min        314.0       360.0  113040.0           0.8            0.0
25%        360.0       360.0  129600.0           1.0            0.0
50%        360.0       360.0  129600.0           1.0            0.0
75%        379.0       360.0  136440.0           1.0            0.0
max        928.0       576.0  501120.0           1.8          388.0

Podejrzane boksy (pole < 10,000 lub > 400,000 px²): 1 (0.00%)
Podejrzane proporcje (AR < 0.25 lub > 2.0):            0  (0.00%)
Segmenty z paddingiem (≥1 bok):                                  7904 (19.9%)

Klucze w train_selection:        33,897
Klucze w LMDB:                   39,813
Pokrycie (LMDB ∩ train):         33,896 (100.00%)

Statystyki jakości kadrowania per osoba migająca (top 15 wg udziału z paddingiem):
             n_segs  mean_area  mean_ar  pct_padded
speaker_id                                         
M_32-45_015      49   124354.3      1.0         1.0
F_32-45_002     197   315259.1      1.0         0.9
M_60+_009       187   147988.9      1.1         0.8
M_46-60_005     186   145651.0      1.1         0.7
M_60+_018        29   142671.7      1.1         0.7
M_32-45_024     270   143169.3      1.1         0.6
M_32-45_023     404   143807.5      1.1         0.6
M_60+_002       202   148168.5      1.1         0.6
M_32-45_003     118   145171.5      1.1         0.6
M_60+_012        29   144446.9      1.1         0.6
M_46-60_017     238   139761.7      1.1         0.6
F_46-60_023      95   136417.3      1.1         0.5
F_46-60_019     266   140768.1      1.1         0.5
M_46-60_012     201   133062.1      1.0         0.5
M_46-60_014     401   147918.7      1.1         0.5


### Faza 6c — Czułość progu słownikowego (min_vocab_pct)

Faza 3 (powyżej) testowała próg `mask_ratio`. Tu badamy, jak zmieniają się:
rozmiar datasetu, liczba unikalnych migających i Gini datasetu
w zależności od progu `min_vocab_pct` (jakość pokrycia słownikowego segmentu).
Hipoteza startowa PRD: próg 80. percentyl.


In [13]:
# min_vocab_pct sensitivity sweep
# Kolumna min_vocab_pct jest w train_df (eksportowana w Fazie 5)
if "min_vocab_pct" not in train_df.columns:
    print("UWAGA: kolumna min_vocab_pct nie jest dostępna w train_selection.csv")
    print("Dostępne kolumny:", list(train_df.columns))
else:
    baseline_p80 = train_df["min_vocab_pct"].quantile(0.80)
    print(f"Baseline hipoteza PRD — 80. percentyl: {baseline_p80:.4f}")
    print(f"Rozkład min_vocab_pct: min={train_df['min_vocab_pct'].min():.4f}, "
          f"median={train_df['min_vocab_pct'].median():.4f}, "
          f"max={train_df['min_vocab_pct'].max():.4f}")
    print()

    thresholds = np.percentile(train_df["min_vocab_pct"].dropna(),
                               [60, 65, 70, 75, 80, 85, 90])

    print(f"{'Próg (percentyl)':>18} {'Próg (wartość)':>15} {'Segmenty':>10} {'Migający':>10} {'Retencja':>10} {'Gini':>8}")
    print("-" * 75)

    rows_sensitivity = []
    for pct, thr in zip([60, 65, 70, 75, 80, 85, 90], thresholds):
        subset = train_df[train_df["min_vocab_pct"] >= thr]
        n_segs = len(subset)
        n_spk  = subset["speaker_id"].nunique()
        retention = n_segs / len(train_df) * 100
        # Gini coefficient on speaker segment counts
        counts = subset["speaker_id"].value_counts().values.astype(float)
        counts_sorted = np.sort(counts)
        n = len(counts_sorted)
        gini = (2 * np.sum((np.arange(1, n+1)) * counts_sorted) / (n * counts_sorted.sum()) - (n+1)/n) if n > 0 else 0
        star = " ← baseline" if pct == 80 else ""
        print(f"{pct:>17}. {thr:>15.4f} {n_segs:>10,} {n_spk:>10} {retention:>9.1f}% {gini:>8.3f}{star}")
        rows_sensitivity.append((pct, thr, n_segs, n_spk, retention, gini))

    # Wykres: retencja vs próg
    pcts_list   = [r[0] for r in rows_sensitivity]
    ret_list    = [r[4] for r in rows_sensitivity]
    spk_list    = [r[3] for r in rows_sensitivity]
    gini_list   = [r[5] for r in rows_sensitivity]

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=["Retencja segmentów vs próg vocab", "Gini (nierównomierność) vs próg vocab"],
    )
    fig.add_trace(go.Scatter(x=pcts_list, y=ret_list, mode="lines+markers", name="Retencja %"), row=1, col=1)
    fig.add_vline(x=80, line_dash="dash", line_color="red", annotation_text="baseline", row=1, col=1)
    fig.add_trace(go.Scatter(x=pcts_list, y=gini_list, mode="lines+markers", name="Gini"), row=1, col=2)
    fig.add_vline(x=80, line_dash="dash", line_color="red", row=1, col=2)
    fig.update_xaxes(title_text="Percentyl progu min_vocab_pct")
    fig.update_layout(title_text="Czułość progu słownikowego", height=380, showlegend=False)
    fig.show()


Baseline hipoteza PRD — 80. percentyl: 96.7700
Rozkład min_vocab_pct: min=21.8300, median=89.9500, max=100.0000

  Próg (percentyl)  Próg (wartość)   Segmenty   Migający   Retencja     Gini
---------------------------------------------------------------------------
               60.         92.5800     13,570        151      40.0%    0.253
               65.         93.8200     11,869        151      35.0%    0.260
               70.         94.9100     10,181        151      30.0%    0.268
               75.         95.8800      8,477        151      25.0%    0.271
               80.         96.7700      6,797        151      20.1%    0.284 ← baseline
               85.         97.6300      5,095        151      15.0%    0.292
               90.         98.4300      3,406        151      10.0%    0.301


### Faza 6d — Grupowy split train/val/test (stratyfikowany per migający)

**Kluczowe wymagania (z PRD):**
- Zero leakage: żadna osoba migająca nie trafia jednocześnie do train i test/val
- Signer-proportion caps: wyniki z Fazy 6a informują o limicie segmentów na osobę
- Proporcje docelowe: np. 80/10/10 lub 85/10/5 — decyzja projektowa
- Reproducibility: stały `random_seed`


In [20]:
# TODO(human): Zaimplementuj speaker-stratified split
#
# Masz do dyspozycji:
#   train_df        — przefiltrowany dataset (33,897 segmentów, 151 migających)
#   speaker_counts  — liczba segmentów per osoba migająca (z Fazy 6a)
#   per_speaker_quality — jakość kadrowania per osoba (z Fazy 6b, jeśli dostępna)
#
# Zdecyduj i zaimplementuj:
#   1. Proporcje podziału (train/val/test) — np. 80/10/10 lub 85/7/8
#   2. Proportion cap z Fazy 6a — ile max segmentów na osobę migającą?
#   3. Strategię przypisania osób do zbiorów:
#      a) Losowe grupowe (shuffle speakers, assign groups) — proste, może dawać
#         nierównomierny rozkład segmentów jeśli sygnerzy mają bardzo różne liczby.
#      b) Stratyfikowane grupowe (sortuj osoby wg liczby segmentów, rozdzielaj
#         round-robin do train/val/test) — lepszy balans rozmiaru zbiorów.
#
# Wyjście: zapisz 3 pliki CSV:
#   ./data/split_train.csv, ./data/split_val.csv, ./data/split_test.csv
# Każdy powinien mieć kolumny co najmniej: key, speaker_id, min_vocab_pct
#
# Po implementacji wydrukuj:
#   - Rozmiary zbiorów (segmenty + unikalne osoby)
#   - Potwierdzenie zero leakage (assert set intersection jest pusty)

#RANDOM_SEED = 42 # dla powtarzalności losowania (jeśli użyjesz losowego shuffle), na razie nie jest potrzebny przy round-robin

speaker_sorted = speaker_counts.sort_values(ascending=False).index.tolist()

assignments = {spk: ["train", "val", "test"][i%3] for i, spk in enumerate(speaker_sorted)}

train_spk = [s for s, a in assignments.items() if a == "train"]
val_spk   = [s for s, a in assignments.items() if a == "val"]
test_spk  = [s for s, a in assignments.items() if a == "test"]


train_split = train_df[train_df["speaker_id"].isin(train_spk)].copy()
val_split = train_df[train_df["speaker_id"].isin(val_spk)].copy()
test_split = train_df[train_df["speaker_id"].isin(test_spk)].copy()

train_split.to_csv("./data/split_train.csv", index=False)
val_split.to_csv("./data/split_val.csv", index=False)
test_split.to_csv("./data/split_test.csv", index=False)

assert set(train_spk) & set(val_spk) == set(), "Leakage: train and val share speakers"
assert set(train_spk) & set(test_spk) == set(), "Leakage: train and test share speakers"
assert set(val_spk) & set(test_spk) == set(), "Leakage: val and test share speakers"
total = len(train_split) + len(val_split) + len(test_split)
assert total == len(train_df), f"Zgubiono segmenty: {len(train_df)} → {total}"

print("Rozmiary zbiorów:")
print(f"  Train: {len(train_split)} segmentów, {train_split['speaker_id'].nunique()} osób")
print(f"  Val:   {len(val_split)} segmentów, {val_split['speaker_id'].nunique()} osób")
print(f"  Test:  {len(test_split)} segmentów, {test_split['speaker_id'].nunique()} osób")
for name, subset in [("train", train_split), ("val", val_split), ("test", test_split)]:                                                                                                                                                     
      pct = crop_train[crop_train["key"].isin(subset["key"])]["has_padding"].mean()                                                                                                                                                           
      print(f"{name}: {pct:.1%} z paddingiem")



Rozmiary zbiorów:
  Train: 11448 segmentów, 51 osób
  Val:   11293 segmentów, 50 osób
  Test:  11156 segmentów, 50 osób
train: 23.2% z paddingiem
val: 21.0% z paddingiem
test: 20.8% z paddingiem
